In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, to_timestamp
from pyspark.sql import functions

my_spark = SparkSession.builder.appName('SalesForecast').getOrCreate()

d:\Programming\Coding UStudy\pyspark_lessons\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
sales_data = my_spark.read.csv("Online Retail.csv", header=True, inferSchema=True)

In [3]:
sales_data.show()

+---------+---------+--------------------+--------+---------+----------+--------------+-------------------+----+-----+----+---+---------+
|InvoiceNo|StockCode|         Description|Quantity|UnitPrice|CustomerID|       Country|        InvoiceDate|Year|Month|Week|Day|DayOfWeek|
+---------+---------+--------------------+--------+---------+----------+--------------+-------------------+----+-----+----+---+---------+
|   536365|   85123A|WHITE HANGING HEA...|       6|     2.55|     17850|United Kingdom|2010-01-12 08:26:00|2010|    1|   2| 12|        1|
|   536365|    71053| WHITE METAL LANTERN|       6|     3.39|     17850|United Kingdom|2010-01-12 08:26:00|2010|    1|   2| 12|        1|
|   536365|   84406B|CREAM CUPID HEART...|       8|     2.75|     17850|United Kingdom|2010-01-12 08:26:00|2010|    1|   2| 12|        1|
|   536365|   84029G|KNITTED UNION FLA...|       6|     3.39|     17850|United Kingdom|2010-01-12 08:26:00|2010|    1|   2| 12|        1|
|   536365|   84029E|RED WOOLLY HO

In [4]:
sales_data = sales_data.withColumn('InvoiceDate', to_date(to_timestamp(col('InvoiceDate'))))

In [5]:
daily_sales_data = sales_data.groupby('Country', 'StockCode', 'InvoiceDate', 'Year', 'Month', 'Week', 'DayOfWeek').agg(functions.sum('Quantity'), functions.sum('UnitPrice'))

In [6]:
daily_sales_data.show()

+--------------+---------+-----------+----+-----+----+---------+-------------+------------------+
|       Country|StockCode|InvoiceDate|Year|Month|Week|DayOfWeek|sum(Quantity)|    sum(UnitPrice)|
+--------------+---------+-----------+----+-----+----+---------+-------------+------------------+
|United Kingdom|    22813| 2010-01-12|2010|    1|   2|        1|           17|              5.85|
|United Kingdom|    22638| 2010-01-12|2010|    1|   2|        1|            1|              2.55|
|United Kingdom|   84997A| 2010-01-12|2010|    1|   2|        1|            6|              3.75|
|United Kingdom|    21071| 2010-02-12|2010|    2|   6|        4|           84|12.910000000000004|
|United Kingdom|    22568| 2010-02-12|2010|    2|   6|        4|           17|              15.0|
|United Kingdom|    20967| 2010-02-12|2010|    2|   6|        4|           10|             18.75|
|United Kingdom|    22576| 2010-02-12|2010|    2|   6|        4|           48|              2.55|
|United Kingdom|    

In [7]:
import pandas as pd
df = pd.read_csv("Online Retail.csv")
df['Year'].value_counts()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 384721 entries, 0 to 384720
Data columns (total 13 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    384721 non-null  int64  
 1   StockCode    384721 non-null  str    
 2   Description  384721 non-null  str    
 3   Quantity     384721 non-null  int64  
 4   UnitPrice    384721 non-null  float64
 5   CustomerID   384721 non-null  int64  
 6   Country      384721 non-null  str    
 7   InvoiceDate  384721 non-null  str    
 8   Year         384721 non-null  int64  
 9   Month        384721 non-null  int64  
 10  Week         384721 non-null  int64  
 11  Day          384721 non-null  int64  
 12  DayOfWeek    384721 non-null  int64  
dtypes: float64(1), int64(8), str(4)
memory usage: 38.2 MB


In [8]:
split_date_train_test = '2011-06-30'

train_data = sales_data.filter(col('InvoiceDate') <= split_date_train_test)

test_data = sales_data.filter(col('InvoiceDate') > split_date_train_test)

from pyspark.ml.feature import StringIndexer, VectorAssembler

country_indexer = StringIndexer(inputCol='Country', outputCol='CountryIndexer').setHandleInvalid('keep')

stock_code_indexer = StringIndexer(inputCol='StockCode', outputCol='StockCodeIndexer').setHandleInvalid('keep')

features_cols = ['CountryIndexer', 'StockCodeIndexer', 'Month', 'Year', 'DayOfWeek', 'Day', 'Week']

assembler = VectorAssembler(inputCols=features_cols, outputCol='Features')

from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

rf = RandomForestRegressor(featuresCol='Features', labelCol='Quantity', maxBins=4000)

In [9]:
pipeline = Pipeline(
    stages=[
        country_indexer, stock_code_indexer, assembler, rf
    ]
)

model = pipeline.fit(train_data)

test_predictions = model.transform(test_data).withColumn('Prediction', col('Prediction').cast('double'))

from pyspark.ml.evaluation import RegressionEvaluator

mae = RegressionEvaluator(labelCol='Quantity', predictionCol='Prediction', metricName='mae')

mae = mae.evaluate(test_predictions)
print('Mean absolute error:', mae)

df = df[(df['Year'] == 2011) & (df['Month']==1)]
df['Day'].value_counts()

Mean absolute error: 5.594980751607315


Day
12    2071
11    1682
9     1324
27    1295
2     1166
25    1154
4     1080
3     1062
8      980
26     976
31     966
17     945
19     906
23     849
7      815
24     739
30     714
13     708
14     690
20     669
6      660
16     634
21     631
28     618
18     506
5      442
Name: count, dtype: int64